# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

##  Unit of analysis + time window

### Unit of Analysis + Time Window

One row represents one anonymized content page observation.

The analysis unit is `content_id` because the business decision is made at the content page level: identifying which pages should receive optimization attention.

The dataset contains one observation per content page, with search performance, engagement, ranking, and content metadata signals.

Time window:

The available dataset is treated as a historical warehouse snapshot provided for the internship task. Since the extracted table does not contain an explicit date column, the analysis uses the available snapshot rather than creating an artificial time window.

The model objective is to rank content pages by optimization opportunity using only information available in the dataset.

The contract separates:

- features available before a decision
- ranking proxy signals
- excluded information that would create leakage

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

##  Fields: Feature / Label / Context / Excluded

### Feature Fields

| Field | Reason |
|---|---|
| search_volume | Measures existing search demand |
| ctr | Measures how effectively impressions generate clicks |
| avg_position | Represents current search ranking visibility |
| engagement_rate | Represents user interaction quality |
| impressions_90d | Measures historical search visibility over the last 90 days |

### Ranking Target / Proxy

The objective is to rank content pages by optimization opportunity.

The dataset does not contain a direct future outcome showing whether an optimization improved performance.

Therefore, the ranking proxy uses observed search intelligence signals such as:

- current ranking position
- search demand
- engagement behavior
- performance trends

The output is a prioritized list of pages rather than a guaranteed prediction of future traffic.

### Context Fields

| Field | Purpose |
|---|---|
| content_id | Identifies each content page |
| client_id | Provides grouping information |
| content_type | Describes page category |

### Excluded Fields

| Field | Reason |
|---|---|
| future optimization results | Not available at decision time |
| post-update metrics | Would introduce leakage |
| client_id as predictive feature | Could cause memorization instead of general patterns |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

# Data Contract Verification

Each assumption in the contract is validated using measurable checks.

The following checks verify:
1. Dataset grain
2. Dataset size and coverage
3. Missing values and data quality

In [36]:
# Query 1: Verify dataset grain
import pandas as pd
total_rows = len(df)
unique_pages = df["content_id"].nunique()

print("Total rows:", total_rows)
print("Unique content pages:", unique_pages)

if total_rows == unique_pages:
    print("Verified: one row represents one unique content page.")
else:
    print("Multiple rows exist per content page.")

Total rows: 30000
Unique content pages: 30000
Verified: one row represents one unique content page.


In [32]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nContent Types")
display(df["content_type"].value_counts())

print("\nTrend Direction")
display(df["trend_direction"].value_counts())

Rows: 30000
Columns: 45

Content Types


,count
content_type,
keyword article,27207
feedly article,2096
comparison article,697



Trend Direction


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


In [33]:
# Query 3: Availability verification

df["available"] = (
    df["search_volume"].notna()
    & df["ctr"].notna()
    & df["avg_position"].notna()
    & df["engagement_rate"].notna()
    & df["trend_pct"].notna()
)

available_df = df[df["available"] == True]

print("Rows available for modeling:", len(available_df))

Rows available for modeling: 25287


In [34]:
features_df = df[
[
"search_volume",
"ctr",
"avg_position",
"engagement_rate",
"impressions_90d"
]].copy()

display(features_df.head())

,search_volume,ctr,avg_position,engagement_rate,impressions_90d
0,10.0,0.76,10.6,5.88,3803
1,90.0,0.05,20.3,0.00,15320
2,0.0,0.09,36.5,0.00,12581
3,10.0,0.49,6.2,1.28,11751
4,0.0,0.13,44.0,0.00,19140


### Feature availability

**search_volume** — Available before the optimization decision because it measures existing search demand.

**ctr** — Available before the decision because it summarizes historical click performance.

**avg_position** — Available before the decision because it reflects the current ranking position.

**engagement_rate** — Available before the decision because it summarizes historical user engagement.

**impressions_90d** — Available before the decision because it is calculated from historical search performance.

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Create a modeling dataframe
model_df = df[
    [
        "search_volume",
        "ctr",
        "avg_position",
        "engagement_rate",
        "impressions_90d",
        "trend_pct"
    ]
].dropna()

# Create a copy for the leakage experiment
leak_df = model_df.copy()

# Intentional leakage feature (copied directly from the target)
leak_df["leaked_label"] = leak_df["trend_pct"]

# Use ONLY the leaked feature
X = leak_df[["leaked_label"]]
y = leak_df["trend_pct"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate
score = model.score(X_test, y_test)

print(f"Leakage score (R²): {score:.4f}")

Leakage score (R²): 1.0000


## Leakage Lesson

The leaked_label feature was intentionally created from the target variable to demonstrate data leakage.

The model achieved an unrealistic perfect score because it received information that would only be available after the prediction period.

This feature was removed from the final feature set.

The final ranking system will only use signals available at the decision moment.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits and Assumptions

Although this dataset provides valuable search intelligence signals, there are several limitations to consider when interpreting model results.

### 1. No direct optimization outcome

The dataset contains observed content performance metrics but does not show the guaranteed impact after a content refresh.

Therefore, the model can rank pages with potential optimization opportunities, but it cannot claim that a recommendation will directly cause improved rankings or traffic.

### 2. Missing and incomplete metadata

Some fields contain missing values, including content metadata and AI-related attributes.

These missing values may represent unavailable information rather than negative performance, so they should be handled carefully during feature engineering.

### 3. External search environment changes

Search performance can change because of factors outside this dataset, including:

- Search algorithm updates
- Competitor activity
- Seasonal demand changes
- Market trends

These external factors are not fully captured by the available features.

### 4. Client and content variation

Different clients and content categories may have different strategies, audiences, and performance patterns.

Client identifiers are treated as context rather than predictive features to reduce the risk of memorizing specific clients.

### Conclusion

The model should be used as a decision-support ranking system that helps prioritize content opportunities. It provides directional insights from observed data, but it does not guarantee future search outcomes.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.